# Scientific Ablation Study: Physiological Loss Functions
**Objective**: Develop a scientifically defensible ECG reconstruction model by systematically introducing loss terms based on signal physiology rather than hyperparameter tuning.

## The Scientific Lifecycle (Phased Approach)

### Phase 0: Baseline Fidelity (Current)
- **Goal**: Establish rigorous baselines ($models$: Mason-CNN, cNVAE-ECG).
- **Loss**: MAE Only (Focus on Signal Amplitude fidelity).
- **Metric**: Validation Loss / RMSE. (Downstream Target: AUROC > 0.88).

### Phase 1: Temporal Dynamics
- **Hypothesis**: Standard MSE/MAE misses rapid depolarization events (QRS complex).
- **Intervention**: Add **First-Derivative Loss** ($L_{grad}$).
- **Validation**: Does AUROC improve? If yes (>0.5%), retain. If no, reject as noise.

### Phase 2: Morphological Topology
- **Hypothesis**: Euclidean distance fails to capture diagnostic shape (ST-Elevation).
- **Intervention**: Add **Pearson Correlation Loss** ($L_{corr}$), but only if Phase 1 justified the complexity.
- **Validation**: Ablation Table comparing P0, P1, P2.


## Theoretical Framework: Why These Losses?

We move beyond "parameter tuning" to **Physiological Modeling**:

| Loss Term | Physiological Justification | Mathematical Role | weight ($\lambda$) |
| :--- | :--- | :--- | :--- |
| **MAE ($L_1$)** | **Isoelectric Stability**. Ensures the baseline voltage is correct and minimizes outlier penalties compared to MSE. | $\mathcal{L}_{MAE} = \frac{1}{N} \sum |y - \hat{y}|$ | $\lambda_1 = 1.0$ |
| **Derivative ($L_{grad}$)** | **Conduction Velocity**. The first derivative of the ECG represents the rate of voltage change ($dV/dt$), critical for detecting QRS onset/offset and conduction blocks. | $\mathcal{L}_{grad} = \frac{1}{N} \sum |\nabla y - \nabla \hat{y}|$ | $\lambda_2 = 1.0$ (Normalized) |
| **Correlation ($L_{corr}$)** | **Diagnostic Morphology**. Clinicians diagnose based on *shape* (e.g., ST elevation), not just absolute amplitude. Pearson correlation allows the model to learn shape invariance. | $\mathcal{L}_{corr} = 1 - \rho(y, \hat{y})$ | $\lambda_3 = 0.5$ (Regularizer) |


In [ ]:
# -----------------------------------------------------------------------------
# PHASE 4: LIVE MONITORING
# -----------------------------------------------------------------------------
import glob
import re
import matplotlib.pyplot as plt
import pandas as pd
import os
import seaborn as sns

sns.set_style("darkgrid")

def parse_ablation_log(logfile):
    data = []
    if not os.path.exists(logfile):
        return pd.DataFrame()
    with open(logfile, 'r') as f:
        for line in f:
            match = re.search(r'Epoch (\d+)/\d+ \| Train: ([\d.]+) \| Val: ([\d.]+)', line)
            if match:
                data.append({
                    'Epoch': int(match.group(1)),
                    'Train': float(match.group(2)),
                    'Val': float(match.group(3))
                })
    return pd.DataFrame(data)

models = ['mason', 'mason_ricker', 'cnvae', 'cnvae_ricker']
phases = ['phase0', 'phase1', 'phase2']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, phase in enumerate(phases):
    ax = axes[i]
    ax.set_title(f"Phase {i}: {phase.upper()} Loss")
    for model in models:
        log_path = f"logs/ablation_{phase}_{model}.log"
        df = parse_ablation_log(log_path)
        if not df.empty:
            ax.plot(df['Epoch'], df['Val'], marker='o', label=f"{model}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation Loss")
    ax.legend()

plt.tight_layout()
plt.show()

## Deep Diagnostics: EDA on Model Behavior

We must critically evaluate *where* the model fails. Low RMSE can hide:
1.  **Lead Bias**: Is it reconstructing V1-V6 well but failing on Limb leads?
2.  **Spectral Loss**: Is it acting as a low-pass filter (blurring QRS)?
3.  **Topology Failure**: Are we preserving the ST-segment slope?

Use the interactive cells below to inspect the latest checkpoints.


In [ ]:
import torch
import numpy as np
import sys
import os
from scipy import signal

# Add src to path
sys.path.append(os.getcwd())
sys.path.append(os.path.join(os.getcwd(), "third_party/cNVAE_ECG/conditional"))
sys.path.append(os.path.join(os.getcwd(), "third_party/ecg_reconstruction"))

from src.reconstruction.learn_functions.wrappers import MasonWrapper, CNVAEReconstructor, BNVAEArgs
from src.data.multi_source_dataset import MultiSourceECGDataset

# Load Data (Validation Set)
# Note: Using small target_len for visualization speed, but real training uses 5000
sources = [{"name": "PTB-XL", "path": "data/ptbxl_tensors", "format": "pt"}]
val_ds = MultiSourceECGDataset(split='val', sources=sources, target_len=5000, normalization='min_max')
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=1, shuffle=True)

def load_checkpoint(model_name, phase):
    device = torch.device('cpu')
    path = f"checkpoints/ablations/{model_name}_{phase}.pt"
    if not os.path.exists(path):
        print(f"Checkpoint not found: {path} (Training might be early)")
        return None
    
    if 'mason' in model_name:
        model = MasonWrapper(device).to(device)
    elif 'cnvae' in model_name:
        args = BNVAEArgs()
        model = CNVAEReconstructor(args, device, num_mixtures=1).to(device)
        
    try:
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"Loaded {model_name} ({phase})")
        model.eval()
        return model
    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        return None

def diagnose_sample(model, sample_idx=None):
    if model is None: return
    
    # Get random sample
    batch = next(iter(val_loader))
    x = batch['input']
    y_true = batch['target']
    
    # Inference
    with torch.no_grad():
        if 'cnvae' in model.__class__.__name__.lower():
             # Handle cNVAE patched return
             y_pred, _, _, _, _ = model(x)
        else:
             y_pred = model(x)
             
    # Plot 12-Lead Overlay
    leads = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
    
    fig, axes = plt.subplots(6, 2, figsize=(15, 12)) # 6x2 grid
    axes = axes.flatten()
    
    y_t = y_true[0].cpu().numpy()
    y_p = y_pred[0].cpu().numpy()
    
    errors = []
    
    for i in range(12):
        ax = axes[i]
        ax.plot(y_t[i], label='Original', color='black', alpha=0.6, linewidth=1)
        ax.plot(y_p[i], label='Recon', color='red', alpha=0.7, linewidth=1)
        
        # Calculate Lead RMSE
        rmse = np.sqrt(np.mean((y_t[i] - y_p[i])**2))
        errors.append(rmse)
        
        ax.set_title(f"{leads[i]} (RMSE: {rmse:.4f})")
        if i == 0: ax.legend()
        ax.axis('off')
        
    plt.suptitle("12-Lead Reconstruction Diagnostics", fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Error Topology & Spectral Analysis
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
    
    # 1. Error Topology (Where is it failing?)
    sns.barplot(x=leads, y=errors, ax=ax1, palette="Reds")
    ax1.set_title("Reconstruction Error by Lead (Topology Check)")
    ax1.set_ylabel("RMSE")
    
    # 2. Spectral Fidelity (Filter Check)
    f, Pxx_den_t = signal.welch(y_t.mean(axis=0), fs=500)
    f, Pxx_den_p = signal.welch(y_p.mean(axis=0), fs=500)
    
    ax2.semilogy(f, Pxx_den_t, label='Original PSD', color='black')
    ax2.semilogy(f, Pxx_den_p, label='Recon PSD', color='red')
    ax2.set_title("Spectral Power Density (Frequency Fidelity)")
    ax2.set_xlabel("Frequency [Hz]")
    ax2.set_ylabel("PSD [V**2/Hz]")
    ax2.legend()
    
    plt.show()

# Example Usage (Uncomment to run)
# model = load_checkpoint('mason', 'phase0')
# diagnose_sample(model)
